# Programming using Numba on GPUs

In this lesson, we will learn how to work with _Numba_. Numba is a Python library that allows you to "translate" Python functions into code that is more efficient, while they can still be called like regular Python functions. This process is called "just-in-time" compilation, or sometimes shortened to JIT.

You can find more about Numba in the documentation: https://numba.readthedocs.io/

We start with our vector example and use Numba to JIT-compile it for the CPU. Next, we use Numba to JIT-compile it for the GPU and go over how to improve the performance of our code on the GPU. Finally, we end with a more challenging example.

## Implementation in Python

We begin with out example of `vector_add`. This takes two vectors (A and B) and returns the pairwise addition of the two. This is just a regular old Python function.

In [3]:
import numpy as np

def vector_add(n, A, B):
  C = []
  for i in range(n):
    result = A[i] + B[i]
    C.append(result)
  return C

n = 1_000_000
A = np.random.rand(n)
B = np.random.rand(n)
print("correct:", np.all(A+B == vector_add(n, A, B)))

%timeit vector_add(n, A, B)

correct: True
284 ms ± 27.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Implementation in Numba on CPU

Next, we can use `@numba.jit` to JIT our code to run on the CPU. Notice how we can keep our original function and only need to add the `@numba.jit` decorator.

In [2]:
import numba

@numba.jit
def vector_add_numba(n, A, B):
  C = []
  for i in range(n):
    C.append(A[i] + B[i])
  return C

n = 1_000_000
A = np.random.rand(n)
B = np.random.rand(n)
print("correct:", np.all(A+B == vector_add_numba(n, A, B)))

%timeit vector_add_numba(n, A, B)

correct: True
41.1 ms ± 1.22 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


## Implementation in Numba on GPU

Now, we are going to run our code on GPU. First, we need to check if our version of Numba actually supports CUDA for Nvidia GPUs. We can also get the name of our GPU.

In [1]:
import numba.cuda

if not numba.cuda.is_available():
  raise Exception("You need a ")

print("CUDA is available:", numba.cuda.is_available())
print("which GPU are we using:", numba.cuda.get_current_device())

CUDA is available: True
which GPU are we using: <CUDA device 0 'b'NVIDIA A100-SXM4-40GB''>


Next, we can use the `@numba.cuda.jit` decorator. Notice how this looks similar to the previous `@numba.jit` decorator, except now we are compiling for the GPU instead of the CPU.

In [4]:
@numba.cuda.jit
def vector_add_kernel(n, A, B):
  C = []
  for i in range(n):
    C.append(A[i] + B[i])
  return C

n = 1_000_000
A = np.random.rand(n)
B = np.random.rand(n)

try:
  %timeit  vector_add_kernel(n, A, B)
except Exception as e:
  print(e)

try:
  %timeit  vector_add_kernel[1,1](n, A, B)
except Exception as e:
  print(e)


Kernel launch configuration was not specified. Use the syntax:

kernel_function[blockspergrid, threadsperblock](arg0, arg1, ..., argn)

See https://numba.readthedocs.io/en/stable/cuda/kernels.html#kernel-invocation for help.


Failed in cuda mode pipeline (step: nopython frontend)
Unknown attribute 'append' of type list(undefined)<iv=None>

File "../../../../scratch-local/sheldens.20694722/ipykernel_3219848/575464675.py", line 5:
<source missing, REPL/exec in use?>

During: typing of get attribute at /scratch-local/sheldens.20694722/ipykernel_3219848/575464675.py (5)

File "../../../../scratch-local/sheldens.20694722/ipykernel_3219848/575464675.py", line 5:
<source missing, REPL/exec in use?>

During: Pass nopython_type_inference


/home/sheldens/.local/lib/python3.13/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


Unfortunately, our code did not run successfully. The first error occurs because Numba requires us to add `[blockspergrid, threadsperblock]`. We will get back to this later. For now, we can add `[1,1]` when calling our function.

However, even after adding `[1,1]`, there is still an error. This time it is the error "Unknown attribute 'append' of type list". This happens because not all Python functions are supported on the GPU. You will find that it is common that not all functionality from Python is available on the GPU. Sometimes you might need to make some changes to make your code compile on the GPU.

For our example, we can work around this by creating an empty array `C` on the host (filled with zeros) and then letting the GPU fill the entries of this array.

In [5]:
import numba.cuda

@numba.cuda.jit
def vector_add_kernel(n, A, B, C):
  for i in range(n):
    C[i] = A[i] + B[i]

n = 1_000_000
A = np.random.rand(n)
B = np.random.rand(n)
C = np.zeros(n)

vector_add_gpu[1,1](n, A, B, C)
print("is correct:", np.all(C == A+B))

%timeit vector_add_kernel[1,1](n, A, B, C)

/home/sheldens/.local/lib/python3.13/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/home/sheldens/.local/lib/python3.13/site-packages/numba/cuda/cudadrv/devicearray.py:887: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


is correct: True
102 ms ± 215 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


## Speeding up the implementation on the GPU

Unfortunately, the GPU is not faster than the CPU yet.

The reason is that the GPU is a parallel device. It has many cores (think hundreds of cores!) that can all do work in parallel. However, currently we launch our function on only a single core. This means that a single core has to loop over `n` and do all the work of adding the numbers, while the other GPU cores are idle.

This is the purpose of the `[1,1]` syntax in Numba. Currently, we are telling Numba that we only want to start a single thread on the GPU that runs our function. Ideally, what we want is to have `n` threads running in parallel. It is common in GPU programming to start with one thread per data element, where each thread performs one action. A thread can get its local identifier using the function `i = numba.cuda.grid(1)`.

For this we can use the syntax `[total_number_of_blocks, threads_per_block]`. This asks Numba to launch `total_number_of_blocks` of something called "thread blocks", with a fixed number of `threads_per_block` threads per thread block. This means that in total, we get `total_number_of_blocks * threads_per_block` threads that all work in parallel.

Play around with the number of thread block and the number of threads per bock.

In [11]:
import numba.cuda

@numba.cuda.jit
def vector_add_parallel_kernel(n, A, B, C):
  i = numba.cuda.grid(1)
  C[i] = A[i] + B[i]


threads_per_block = 1000
total_number_of_blocks = 1000
n = total_number_of_blocks * threads_per_block

A = np.random.rand(n)
B = np.random.rand(n)
C = np.zeros(n)

vector_add_parallel_kernel[total_number_of_blocks, threads_per_block](n, A, B, C)
print("is correct:", np.all(A + B == C))

%timeit vector_add_parallel_kernel[total_number_of_blocks, threads_per_block](n, A, B, C)

/home/sheldens/.local/lib/python3.13/site-packages/numba/cuda/cudadrv/devicearray.py:887: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


is correct: True
6.74 ms ± 30.7 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


# Memory

Our GPU kernel is still not that fast. The reason for this is not the kernel itself, but instead, the way that we call the kernel. 

Currently, our kernel takes NumPy arrays as arguments. However, this means that each time we call a kernel, Numba has to go through three steps: 
* Copy the NumPy arguments from Python to the GPU 
* Call the GPU kernel
* Copy the GPU results back to NumPy arguments in Python

The first and the third step take a lot of time. If we call multiple kernels in succession, it can even be inefficient if the output result of one kernel will be the input for the next kernel.

Instead, we can also manually move the data from Python to the GPU using `numba.cuda.to_device`. Finally, we can move the results from the GPU back to Python using `copy_to_host`. 

In [32]:
threads_per_block = 1000
number_of_blocks = 100_000
n = number_of_blocks * threads_per_block

A = np.random.rand(n)
B = np.random.rand(n)
C = np.zeros(n)

A_gpu = numba.cuda.to_device(A)
B_gpu = numba.cuda.to_device(B)
C_gpu = numba.cuda.to_device(C)

vector_add_parallel_kernel[number_of_blocks, threads_per_block](n, A_gpu, B_gpu, C_gpu)
%timeit vector_add_parallel_kernel[number_of_blocks, threads_per_block](n, A_gpu, B_gpu, C_gpu); numba.cuda.synchronize()

C = C_gpu.copy_to_host()

print("correct:", np.all(A + B == C))

1.81 ms ± 290 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
correct: True


## Example using `vectorize` (Optional)

There is also the decorator `@numba.vectorize(..., target="cuda")` that you can use if you have a simple function that you want to apply element-wise. Please read more about it here: https://numba.readthedocs.io/en/stable/cuda/ufunc.html

In [ ]:
import numpy as np
import numba

@numba.vectorize(["float64(float64, float64)"], target="cuda")
def vector_add_vectorize(a, b):
    return a + b

n = 1_000_000

A = np.random.rand(n)
B = np.random.rand(n)

%timeit C = vector_add_vectorize(A_gpu, B_gpu)
print(type(C))

# Example with computing prime numbers (Optional)

Next, let's try a more complex example. First, we create function called `is_prime` that returns `True` if a given number is a prime number, otherwise it returns `False`.

def is_prime(number):
  if number < 2:
    return False
  for k in range(2, number):
    if number % k == 0:
      return False
  return True

print("is prime 1:", is_prime(1))
print("is prime 2:", is_prime(2))
print("is prime 3:", is_prime(3))
print("is prime 10:", is_prime(10))
print("is prime 12345:", is_prime(12345))

Next, we create a simple Python function that takes an array of numbers and returns an array of booleans that indicates if each number is a prime number or note.

In [41]:
def find_primes(numbers):
  results = []
  for number in numbers:
    results.append(is_prime(number))
  return results

n = 10_000
numbers = np.arange(n)
%timeit find_primes(numbers)

768 ms ± 39.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


Now, try to use Numba to speed this up! How fast can you get? How can you make this work parallel?

## Answer: CPU

In [45]:
import numba

@numba.jit
def is_prime_numba(number):
  if number < 2:
    return False
  for k in range(2, number):
    if number % k == 0:
      return False
  return True

@numba.jit
def find_primes_numba(numbers):
  results = []
  for number in numbers:
    results.append(is_prime_numba(number))
  return results

n = 100000
numbers = np.arange(n)
%timeit find_primes_numba(numbers)

1.16 s ± 747 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Answer: GPU

In [ ]:
import numba.cuda

@numba.cuda.jit
def find_primes_cuda(numbers, results):
  i = numba.cuda.grid(1)
  results[i] = is_prime_numba(numbers[i])

num_blocks = 1000
threads_per_block = 100
n = threads_per_block * num_blocks

numbers = np.arange(n)
results = np.zeros(n)
%timeit find_primes_cuda[num_blocks, threads_per_block](numbers, results)